# OpenAI의 Embedding model 을 활용한 코사인 유사도 검색 시스템

In [ ]:
import os
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings

import config


In [ ]:
# "저는 배가 고파요" 라는 텍스트를 테스트 임베딩

embeddings = OpenAIEmbeddings(model=config.EMBEDDING_MODEL, openai_api_key=config.OPENAI_API_KEY)
query_result = embeddings.embed_query("저는 배가 고파요")
print(query_result)

[0.017364501953125, -0.0078582763671875, -0.0290985107421875, 0.0075531005859375, 0.00115203857421875, -0.058746337890625, -0.033721923828125, 0.033599853515625, -0.0467529296875, -0.01544189453125, -0.02069091796875, 0.051239013671875, -0.0013132095336914062, -0.0038471221923828125, -0.0169677734375, 0.002384185791015625, -0.05816650390625, 0.00034999847412109375, 0.004978179931640625, -0.03277587890625, -0.0245513916015625, -0.0157623291015625, -0.023284912109375, -0.006778717041015625, 0.0020580291748046875, -0.00034737586975097656, 0.039581298828125, -0.033721923828125, 0.0036334991455078125, -0.01157379150390625, 0.00989532470703125, -0.04766845703125, 0.0180511474609375, -0.0484619140625, -0.01898193359375, -0.0386962890625, 0.011688232421875, -0.037689208984375, -0.00418853759765625, 0.0011758804321289062, -0.029205322265625, 0.0015573501586914062, -0.0179901123046875, 0.008941650390625, 0.03289794921875, 0.036041259765625, -0.005748748779296875, 0.01514434814453125, 0.051727294

In [ ]:
# 6개의 문장을 임베딩해 pandas 표로 출력

data = ['저는 배가 고파요',
        '저기 배가 지나가네요',
        '굶어서 허기가 지네요',
        '허기 워기라는 게임이 있는데 즐거워',
        '스팀에서 재밌는 거 해야지',
        '스팀에어프라이어로 연어구이 해먹을거야']

df = pd.DataFrame(data, columns=['text'])
df

,text
0,저는 배가 고파요
1,저기 배가 지나가네요
2,굶어서 허기가 지네요
3,허기 워기라는 게임이 있는데 즐거워
4,스팀에서 재밌는 거 해야지
5,스팀에어프라이어로 연어구이 해먹을거야


In [ ]:
# 문장을 임베딩 해주는 get_embedding 함수와 df 데이터를 행 단위로 embedding apply

def get_embedding(text):
    return embeddings.embed_query(text)

df['embedding'] = df.apply(lambda row:get_embedding(row.text), axis=1)
df

,text,embedding
0,저는 배가 고파요,"[0.017364501953125, -0.0078582763671875, -0.02..."
1,저기 배가 지나가네요,"[0.03948974609375, 8.654594421386719e-05, -0.0..."
2,굶어서 허기가 지네요,"[0.041534423828125, 0.0248260498046875, -0.012..."
3,허기 워기라는 게임이 있는데 즐거워,"[0.00748443603515625, -0.00396728515625, -0.06..."
4,스팀에서 재밌는 거 해야지,"[0.01322174072265625, 0.05426025390625, -0.040..."
5,스팀에어프라이어로 연어구이 해먹을거야,"[0.050018310546875, 0.0038051605224609375, 0.0..."


In [ ]:
# 코사인 유사도를 출력하는 함수 cos_sim 과, df 와 query(입력값) 의 유사도를 비교하는 함수 (내림차순 3개 출력)

def cos_sim(A,B):
    return dot(A, B)/(norm(A)*norm(B))

def return_answer_candidate(df, query):
    query_embedding = get_embedding(query)
    df["similarity"] = df.embedding.apply(
        lambda x:cos_sim(np.array(x), np.array(query_embedding)))
    top_three_doc = df.sort_values("similarity", ascending=False).head(3)
    return top_three_doc

In [ ]:
# 테스트용 문장을 입력하고 유사도가 높은 문장을 확인

sim_result = return_answer_candidate(df,'아무것도 안 먹었더니 꼬르륵 소리가 나네')
sim_result

,text,embedding,similarity
5,스팀에어프라이어로 연어구이 해먹을거야,"[0.050018310546875, 0.0038051605224609375, 0.0...",0.322957
0,저는 배가 고파요,"[0.017364501953125, -0.0078582763671875, -0.02...",0.247882
4,스팀에서 재밌는 거 해야지,"[0.01322174072265625, 0.05426025390625, -0.040...",0.225850


In [ ]:
# 두가지 모델을 비교해 높은 유사도를 확인해 보았음.

# 5 0 4 [text-embedding-3-small]
# 2 5 0 [text-embedding-ada-002]